# 04 · Build paired pseudobulk samples

Attach the Stage 03 H5AD and manifest. Sum raw UMI counts within each donor and condition, then apply a pre-DE expression filter. Stage 05 uses both the full and filtered matrices to verify gene selection.


## 1. Input provenance and integrity

Verify the Stage 03 manifest, H5AD digest and cohort dimensions.


In [1]:

from pathlib import Path
import hashlib
import json

# --------------------------------------------------
# 1. Locate the Stage 03 output
# --------------------------------------------------

INPUT_DIR = Path("/kaggle/input")

candidates = sorted(
    INPUT_DIR.rglob("ifnb_cd14_candidate_qc.h5ad")
)

print("Candidate H5AD files found:", len(candidates))

for p in candidates:
    print(" -", p)

assert len(candidates) == 1, (
    "Expected exactly one Stage 03 H5AD. "
    "Check Kaggle Add Input and attached versions."
)

H5AD_PATH = candidates[0]

# Output layout:
# kang_ifnb/data/processed/<file>
SOURCE_PROJECT = H5AD_PATH.parents[2]

MANIFEST_PATH = (
    SOURCE_PROJECT
    / "results"
    / "audit"
    / "stage03_cd14_h5ad_roundtrip.json"
)

assert MANIFEST_PATH.is_file(), (
    f"Missing Stage 03 manifest: {MANIFEST_PATH}"
)

# --------------------------------------------------
# 2. Read the recorded manifest
# --------------------------------------------------

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

EXPECTED_SHA256 = (
    "8747d0741e1197f9a2f0dff26817bf803ebcf36dca7bd6f5806f7350da88455d"
)

assert manifest["sha256"] == EXPECTED_SHA256
assert manifest["shape"] == [5357, 35635]
assert manifest["total_umi"] == 12580095
assert manifest["nnz"] == 3700565
assert manifest["roundtrip"] == "PASS"

# --------------------------------------------------
# 3. Independently compute checksum
# --------------------------------------------------

sha256 = hashlib.sha256()

with H5AD_PATH.open("rb") as f:
    for chunk in iter(
        lambda: f.read(8 * 1024 * 1024),
        b"",
    ):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

assert actual_sha256 == EXPECTED_SHA256
assert H5AD_PATH.stat().st_size == manifest[
    "file_size_bytes"
]

# --------------------------------------------------
# 4. Define Stage 04 output directories
# --------------------------------------------------

PROJECT = Path("/kaggle/working/kang_ifnb")

PROCESSED_DIR = PROJECT / "data" / "processed"
AUDIT_DIR = PROJECT / "results" / "audit"
FIGURES_DIR = PROJECT / "results" / "figures"

for directory in [
    PROCESSED_DIR, AUDIT_DIR, FIGURES_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\nInput H5AD:", H5AD_PATH)
print("Source manifest:", MANIFEST_PATH)
print("SHA256:", actual_sha256)

print("\nSTAGE 04 INPUT CHECKPOINT: PASS")


Candidate H5AD files found: 1
 - /kaggle/input/notebooks/tthanhh811/03-ifnb-qc-ipynb/kang_ifnb/data/processed/ifnb_cd14_candidate_qc.h5ad

Input H5AD: /kaggle/input/notebooks/tthanhh811/03-ifnb-qc-ipynb/kang_ifnb/data/processed/ifnb_cd14_candidate_qc.h5ad
Source manifest: /kaggle/input/notebooks/tthanhh811/03-ifnb-qc-ipynb/kang_ifnb/results/audit/stage03_cd14_h5ad_roundtrip.json
SHA256: 8747d0741e1197f9a2f0dff26817bf803ebcf36dca7bd6f5806f7350da88455d

STAGE 04 INPUT CHECKPOINT: PASS


In [2]:
%pip -q install "anndata==0.13.4"

In [3]:

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from importlib.metadata import version

# --------------------------------------------------
# 1. Verify runtime and load AnnData
# --------------------------------------------------

assert version("anndata") == "0.13.4"

adata_cd14 = ad.read_h5ad(H5AD_PATH)

assert adata_cd14.shape == (5357, 35635)
assert adata_cd14.raw is None

# --------------------------------------------------
# 2. Verify the raw integer count matrices
# --------------------------------------------------

counts = adata_cd14.layers["counts"]

assert sparse.isspmatrix_csr(counts)
assert sparse.isspmatrix_csr(adata_cd14.X)

assert counts.dtype == np.int64
assert adata_cd14.X.dtype == np.int64

assert counts.nnz == 3700565
assert int(counts.sum()) == 12580095

assert (
    counts != adata_cd14.X
).nnz == 0

# Existing per-cell UMI totals must reconcile
assert np.array_equal(
    adata_cd14.obs["total_counts"].to_numpy(
        dtype=np.int64
    ),
    np.asarray(counts.sum(axis=1)).ravel(),
)

# --------------------------------------------------
# 3. Verify identifiers and cohort definition
# --------------------------------------------------

assert adata_cd14.obs_names.is_unique
assert adata_cd14.var_names.is_unique

assert (
    adata_cd14.obs["demuxlet_class"]
    .astype(str).eq("singlet").all()
)

assert (
    adata_cd14.obs["cell_type_reference"]
    .astype(str).eq("CD14+ Monocytes").all()
)

assert (
    adata_cd14.uns["cohort_definition"]["status"]
    == "candidate_pre_DE"
)

# --------------------------------------------------
# 4. Verify the paired donor-condition design
# --------------------------------------------------

obs = adata_cd14.obs[
    ["donor_id", "condition", "source_sample_id"]
].copy()

for col in obs.columns:
    obs[col] = obs[col].astype(str)

assert set(obs["condition"]) == {"ctrl", "stim"}

# Condition-to-source mapping must remain unchanged
assert set(
    obs.loc[
        obs["condition"] == "ctrl",
        "source_sample_id",
    ]
) == {"GSM2560248"}

assert set(
    obs.loc[
        obs["condition"] == "stim",
        "source_sample_id",
    ]
) == {"GSM2560249"}

pairs = pd.crosstab(
    obs["donor_id"],
    obs["condition"],
).reindex(columns=["ctrl", "stim"])

assert pairs.shape == (8, 2)
assert (pairs > 0).all().all()
assert int(pairs.to_numpy().sum()) == 5357

print("AnnData version:", version("anndata"))
print("Loaded shape:", adata_cd14.shape)
print("Count dtype:", counts.dtype)
print("Total UMI:", int(counts.sum()))
print("Nonzero entries:", counts.nnz)

print("\nPaired donor-condition cells:")
display(pairs)

print("\nSTAGE 04 ANNDATA LOADING: PASS")


AnnData version: 0.13.4
Loaded shape: (5357, 35635)
Count dtype: int64
Total UMI: 12580095
Nonzero entries: 3700565

Paired donor-condition cells:


condition,ctrl,stim
donor_id,,
101,202,253
1015,779,671
1016,369,329
1039,112,151
107,209,169
1244,415,290
1256,390,375
1488,295,348



STAGE 04 ANNDATA LOADING: PASS


## 2. Aggregate raw UMI counts by donor and condition

Sum raw counts by donor and condition. Check cell and UMI conservation.


In [4]:

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

# --------------------------------------------------
# 1. Verify cell-level raw counts
# --------------------------------------------------

cell_counts = adata_cd14.layers["counts"]

assert sparse.isspmatrix_csr(cell_counts)
assert cell_counts.dtype == np.int64
assert cell_counts.shape == (5357, 35635)
assert int(cell_counts.sum()) == 12580095

# --------------------------------------------------
# 2. Extract grouping metadata
# --------------------------------------------------

donors = (
    adata_cd14.obs["donor_id"]
    .astype(str)
    .to_numpy()
)

conditions = (
    adata_cd14.obs["condition"]
    .astype(str)
    .to_numpy()
)

source_samples = (
    adata_cd14.obs["source_sample_id"]
    .astype(str)
    .to_numpy()
)

donor_ids = sorted(set(donors), key=int)
condition_order = ["ctrl", "stim"]

expected_source = {
    "ctrl": "GSM2560248",
    "stim": "GSM2560249",
}

assert len(donor_ids) == 8
assert set(conditions) == set(condition_order)

# --------------------------------------------------
# 3. Aggregate each donor-condition group
# --------------------------------------------------

pb_rows = []
sample_records = []

for donor in donor_ids:
    for condition in condition_order:

        selected_rows = np.flatnonzero(
            (donors == donor)
            & (conditions == condition)
        )

        assert len(selected_rows) > 0

        # Check source-capture provenance
        assert set(source_samples[selected_rows]) == {
            expected_source[condition]
        }

        # Sum raw UMI counts gene by gene
        gene_sums = np.asarray(
            cell_counts[selected_rows, :].sum(axis=0)
        ).ravel().astype(np.int64)

        assert gene_sums.shape == (35635,)
        assert (gene_sums >= 0).all()

        # Check conservation within this group
        group_umi = int(gene_sums.sum())

        original_group_umi = int(
            adata_cd14.obs.iloc[selected_rows][
                "total_counts"
            ].sum()
        )

        assert group_umi == original_group_umi

        pb_rows.append(gene_sums)

        sample_records.append({
            "sample_id": f"{donor}__{condition}",
            "donor_id": donor,
            "condition": condition,
            "source_sample_id": expected_source[condition],
            "n_cells": int(len(selected_rows)),
            "library_size_umi": group_umi,
            "genes_with_1plus_umi": int(
                np.count_nonzero(gene_sums >= 1)
            ),
            "genes_with_10plus_umi": int(
                np.count_nonzero(gene_sums >= 10)
            ),
        })

# --------------------------------------------------
# 4. Construct pseudobulk count matrix
# --------------------------------------------------

pb_matrix = sparse.csr_matrix(
    np.vstack(pb_rows),
    dtype=np.int64,
)

assert pb_matrix.shape == (16, 35635)
assert pb_matrix.dtype == np.int64

# --------------------------------------------------
# 5. Construct sample-level metadata
# --------------------------------------------------

pb_obs = (
    pd.DataFrame(sample_records)
    .set_index("sample_id")
)

pb_obs["condition"] = pd.Categorical(
    pb_obs["condition"],
    categories=["ctrl", "stim"],
)

assert pb_obs.index.is_unique
assert len(pb_obs) == 16

# --------------------------------------------------
# 6. Create a separate AnnData object
# --------------------------------------------------

adata_pb = ad.AnnData(
    X=pb_matrix,
    obs=pb_obs,
    var=adata_cd14.var.copy(),
    layers={
        "counts": pb_matrix.copy(),
    },
)

adata_pb.uns["pseudobulk_provenance"] = {
    "source_artifact": "ifnb_cd14_candidate_qc.h5ad",
    "source_cohort_status": "candidate_pre_DE",
    "aggregation_method": "sum_raw_umi",
    "grouping_variables": ["donor_id", "condition"],
    "n_input_cells": 5357,
    "n_pseudobulk_samples": 16,
    "n_paired_donors": 8,
    "gene_filtering_applied": False,
    "normalization_applied": False,
    "status": "aggregated_unfiltered",
    "design_limitation": (
        "Condition and source capture are confounded."
    ),
}

# --------------------------------------------------
# 7. Show result
# --------------------------------------------------

print(adata_pb)

print("\nPseudobulk sample metadata:")
display(adata_pb.obs)

print("\nPSEUDOBULK AGGREGATION: PASS")


AnnData object with n_obs × n_vars = 16 × 35635
    obs: 'donor_id', 'condition', 'source_sample_id', 'n_cells', 'library_size_umi', 'genes_with_1plus_umi', 'genes_with_10plus_umi'
    var: 'gene_symbol', 'mt'
    uns: 'pseudobulk_provenance'
    layers: 'counts', None (.X)

Pseudobulk sample metadata:


,donor_id,condition,source_sample_id,n_cells,library_size_umi,genes_with_1plus_umi,genes_with_10plus_umi
sample_id,,,,,,,
101__ctrl,101,ctrl,GSM2560248,202,524723,9315,3592
101__stim,101,stim,GSM2560249,253,726512,9211,3608
107__ctrl,107,ctrl,GSM2560248,209,469563,8919,3107
107__stim,107,stim,GSM2560249,169,436403,7913,2539
1015__ctrl,1015,ctrl,GSM2560248,779,1685517,11439,6113
1015__stim,1015,stim,GSM2560249,671,1681881,10690,5082
1016__ctrl,1016,ctrl,GSM2560248,369,721958,10100,4293
1016__stim,1016,stim,GSM2560249,329,707388,9356,3606
1039__ctrl,1039,ctrl,GSM2560248,112,270440,7998,2240



PSEUDOBULK AGGREGATION: PASS


In [5]:

# --------------------------------------------------
# 1. Verify AnnData structure
# --------------------------------------------------

assert adata_pb.shape == (16, 35635)

assert sparse.isspmatrix_csr(adata_pb.X)
assert sparse.isspmatrix_csr(
    adata_pb.layers["counts"]
)

assert adata_pb.X.dtype == np.int64
assert adata_pb.layers["counts"].dtype == np.int64

assert adata_pb.raw is None

# Gene IDs and annotations must be preserved
assert adata_pb.var_names.equals(
    adata_cd14.var_names
)

pd.testing.assert_frame_equal(
    adata_pb.var,
    adata_cd14.var,
    check_exact=True,
)

# --------------------------------------------------
# 2. Verify each pseudobulk sample independently
# --------------------------------------------------

for sample_id in adata_pb.obs_names:

    row = adata_pb.obs.loc[sample_id]

    donor = str(row["donor_id"])
    condition = str(row["condition"])

    selected_rows = np.flatnonzero(
        (donors == donor)
        & (conditions == condition)
    )

    expected_counts = np.asarray(
        cell_counts[selected_rows, :].sum(axis=0)
    ).ravel()

    actual_counts = (
        adata_pb[sample_id, :].X
        .toarray()
        .ravel()
    )

    np.testing.assert_array_equal(
        actual_counts,
        expected_counts,
    )

    assert int(row["n_cells"]) == len(
        selected_rows
    )

    assert int(row["library_size_umi"]) == int(
        expected_counts.sum()
    )

print("All 16 sample-level count vectors: EXACT MATCH")

# --------------------------------------------------
# 3. Verify every gene across the entire cohort
# --------------------------------------------------

original_gene_totals = np.asarray(
    cell_counts.sum(axis=0)
).ravel()

pseudobulk_gene_totals = np.asarray(
    adata_pb.X.sum(axis=0)
).ravel()

np.testing.assert_array_equal(
    pseudobulk_gene_totals,
    original_gene_totals,
)

print("All 35,635 gene totals: EXACT MATCH")

# --------------------------------------------------
# 4. Verify global count conservation
# --------------------------------------------------

original_total = int(cell_counts.sum())
pseudobulk_total = int(adata_pb.X.sum())

assert original_total == 12580095
assert pseudobulk_total == 12580095

assert int(adata_pb.obs[
    "library_size_umi"
].sum()) == 12580095

assert int(adata_pb.obs["n_cells"].sum()) == 5357

# X and raw-count layer are identical
assert (
    adata_pb.X != adata_pb.layers["counts"]
).nnz == 0

# --------------------------------------------------
# 5. Verify paired donor structure
# --------------------------------------------------

paired_counts = (
    adata_pb.obs
    .pivot(
        index="donor_id",
        columns="condition",
        values="n_cells",
    )
    .loc[donor_ids, ["ctrl", "stim"]]
)

assert paired_counts.shape == (8, 2)
assert (paired_counts > 0).all().all()

# --------------------------------------------------
# 6. Inspect aggregation characteristics
# --------------------------------------------------

n_expressed_genes = int(
    np.count_nonzero(
        pseudobulk_gene_totals > 0
    )
)

n_all_zero_genes = (
    adata_pb.n_vars - n_expressed_genes
)

print("\nPaired donor cell counts:")
display(paired_counts)

print("\nPseudobulk matrix:", adata_pb.shape)
print("Count dtype:", adata_pb.X.dtype)
print("Nonzero entries:", adata_pb.X.nnz)
print("Total UMI:", pseudobulk_total)

print("Genes with >=1 UMI overall:", n_expressed_genes)
print("Genes with zero UMI overall:", n_all_zero_genes)

# --------------------------------------------------
# 7. Verify source objects remain unchanged
# --------------------------------------------------

assert adata_cd14.shape == (5357, 35635)
assert cell_counts.nnz == 3700565
assert int(cell_counts.sum()) == 12580095

assert adata_pb.uns[
    "pseudobulk_provenance"
]["normalization_applied"] is False

assert adata_pb.uns[
    "pseudobulk_provenance"
]["gene_filtering_applied"] is False

print("\nCells removed in this step: 0")
print("Genes filtered in this step: 0")

print("\nPSEUDOBULK COUNT CONSERVATION: PASS")


All 16 sample-level count vectors: EXACT MATCH
All 35,635 gene totals: EXACT MATCH

Paired donor cell counts:


condition,ctrl,stim
donor_id,,
101,202,253
107,209,169
1015,779,671
1016,369,329
1039,112,151
1244,415,290
1256,390,375
1488,295,348



Pseudobulk matrix: (16, 35635)
Count dtype: int64
Nonzero entries: 151159
Total UMI: 12580095
Genes with >=1 UMI overall: 14814
Genes with zero UMI overall: 20821

Cells removed in this step: 0
Genes filtered in this step: 0

PSEUDOBULK COUNT CONSERVATION: PASS


## 3. Gene abundance and prevalence diagnostics

Summarize gene detection across samples and donor pairs.


In [6]:

import numpy as np
import pandas as pd
from scipy import sparse

# --------------------------------------------------
# 1. Verify pseudobulk matrix
# --------------------------------------------------

pb_counts = adata_pb.layers["counts"]

assert sparse.isspmatrix_csr(pb_counts)
assert pb_counts.dtype == np.int64
assert pb_counts.shape == (16, 35635)
assert pb_counts.nnz == 151159
assert int(pb_counts.sum()) == 12580095

# Only 16 x 35,635 values (~4.6 MB);
# this is NOT the cell-level matrix.
count_array = pb_counts.toarray()

assert count_array.shape == (16, 35635)

# --------------------------------------------------
# 2. Check sample metadata and library sizes
# --------------------------------------------------

sample_donors = (
    adata_pb.obs["donor_id"]
    .astype(str)
    .to_numpy()
)

sample_conditions = (
    adata_pb.obs["condition"]
    .astype(str)
    .to_numpy()
)

lib_sizes = (
    adata_pb.obs["library_size_umi"]
    .to_numpy(dtype=np.int64)
)

np.testing.assert_array_equal(
    count_array.sum(axis=1),
    lib_sizes,
)

assert (lib_sizes > 0).all()

ctrl_rows = sample_conditions == "ctrl"
stim_rows = sample_conditions == "stim"

assert ctrl_rows.sum() == 8
assert stim_rows.sum() == 8

# --------------------------------------------------
# 3. Measure gene abundance and prevalence
# --------------------------------------------------

detected = count_array > 0

gene_diag = pd.DataFrame({
    "gene_id": adata_pb.var_names.astype(str),
    "gene_symbol": (
        adata_pb.var["gene_symbol"]
        .astype(str)
        .to_numpy()
    ),
    "total_umi": count_array.sum(axis=0),
    "n_samples_detected": detected.sum(axis=0),
    "n_samples_ge10_umi": (
        count_array >= 10
    ).sum(axis=0),
    "n_ctrl_detected": (
        detected[ctrl_rows].sum(axis=0)
    ),
    "n_stim_detected": (
        detected[stim_rows].sum(axis=0)
    ),
})

# --------------------------------------------------
# 4. Count paired donors with detection
# --------------------------------------------------

donor_ids = sorted(
    set(sample_donors),
    key=int,
)

n_pairs_both = np.zeros(
    adata_pb.n_vars,
    dtype=np.int64,
)

n_donors_either = np.zeros(
    adata_pb.n_vars,
    dtype=np.int64,
)

for donor in donor_ids:

    ctrl_idx = np.flatnonzero(
        (sample_donors == donor)
        & ctrl_rows
    )

    stim_idx = np.flatnonzero(
        (sample_donors == donor)
        & stim_rows
    )

    assert len(ctrl_idx) == 1
    assert len(stim_idx) == 1

    ctrl_detected = detected[ctrl_idx[0]]
    stim_detected = detected[stim_idx[0]]

    n_pairs_both += (
        ctrl_detected & stim_detected
    )

    n_donors_either += (
        ctrl_detected | stim_detected
    )

gene_diag["n_donor_pairs_both"] = n_pairs_both
gene_diag["n_donors_either"] = n_donors_either

# --------------------------------------------------
# 5. Validate counts and detection frequencies
# --------------------------------------------------

assert len(gene_diag) == 35635

assert int(gene_diag["total_umi"].sum()) == 12580095

assert int(
    gene_diag["n_samples_detected"].sum()
) == 151159

assert int(
    (gene_diag["total_umi"] == 0).sum()
) == 20821

assert int(
    (gene_diag["total_umi"] > 0).sum()
) == 14814

assert (
    gene_diag["n_samples_detected"]
    == (
        gene_diag["n_ctrl_detected"]
        + gene_diag["n_stim_detected"]
    )
).all()

assert (
    gene_diag["n_donor_pairs_both"]
    <= gene_diag["n_donors_either"]
).all()

assert (
    gene_diag["n_donors_either"] <= 8
).all()

print("All genes:", len(gene_diag))
print(
    "Genes with >=1 UMI:",
    int((gene_diag["total_umi"] > 0).sum())
)
print(
    "All-zero genes:",
    int((gene_diag["total_umi"] == 0).sum())
)

print("\nGENE PREVALENCE VALIDATION: PASS")


All genes: 35635
Genes with >=1 UMI: 14814
All-zero genes: 20821

GENE PREVALENCE VALIDATION: PASS


In [7]:

# --------------------------------------------------
# 1. Distribution of gene detection frequency
# --------------------------------------------------

prevalence_distribution = (
    gene_diag["n_samples_detected"]
    .value_counts()
    .reindex(range(17), fill_value=0)
    .sort_index()
    .rename_axis("n_samples_detected")
    .reset_index(name="n_genes")
)

assert prevalence_distribution["n_genes"].sum() == 35635

print("Number of genes by detected pseudobulk samples:")
display(prevalence_distribution)

# --------------------------------------------------
# 2. Descriptive condition-presence categories
# --------------------------------------------------

ctrl_present = gene_diag["n_ctrl_detected"] > 0
stim_present = gene_diag["n_stim_detected"] > 0

presence_categories = pd.Series(
    np.select(
        [
            ~ctrl_present & ~stim_present,
            ctrl_present & ~stim_present,
            ~ctrl_present & stim_present,
            ctrl_present & stim_present,
        ],
        [
            "zero_both",
            "ctrl_only",
            "stim_only",
            "both_conditions",
        ],
        default="unexpected",
    ),
    name="presence_category",
)

presence_summary = (
    presence_categories
    .value_counts()
    .rename_axis("presence_category")
    .reset_index(name="n_genes")
)

assert presence_summary["n_genes"].sum() == 35635

assert (
    presence_summary.loc[
        presence_summary["presence_category"]
        == "zero_both",
        "n_genes",
    ].iloc[0] == 20821
)

print("\nGene detection across conditions:")
display(presence_summary)

# --------------------------------------------------
# 3. CPM as a DIAGNOSTIC only
# --------------------------------------------------

median_library_size = float(
    np.median(lib_sizes)
)

cpm_cutoff = (
    10.0 / median_library_size * 1e6
)

cpm = (
    count_array.astype(np.float64)
    / lib_sizes[:, None]
    * 1e6
)

n_samples_above_cpm = (
    cpm >= cpm_cutoff
).sum(axis=0)

total_umi_per_gene = (
    gene_diag["total_umi"].to_numpy()
)

# --------------------------------------------------
# 4. Compare exploratory minimum sample support
# --------------------------------------------------

preview_records = []

for min_samples in [2, 4, 8]:

    candidate = (
        (n_samples_above_cpm >= min_samples)
        & (total_umi_per_gene >= 15)
    )

    preview_records.append({
        "min_samples": min_samples,
        "cpm_cutoff": cpm_cutoff,
        "min_total_umi": 15,
        "n_candidate_genes": int(candidate.sum()),
        "n_not_selected": int((~candidate).sum()),
    })

filter_preview = pd.DataFrame(preview_records)

assert (
    filter_preview["n_candidate_genes"]
    .diff()
    .dropna()
    <= 0
).all()

print("\nMedian pseudobulk library size:",
      median_library_size)

print("Exploratory CPM cutoff:",
      round(cpm_cutoff, 3))

print("\nExploratory gene-filtering preview:")
display(filter_preview.round(3))

# --------------------------------------------------
# 5. Save reproducible diagnostics
# --------------------------------------------------

gene_diag["n_samples_above_cpm_cutoff"] = (
    n_samples_above_cpm
)

gene_diag.to_csv(
    AUDIT_DIR / "stage04_gene_prevalence.tsv.gz",
    sep="\t",
    index=False,
)

prevalence_distribution.to_csv(
    AUDIT_DIR / "stage04_gene_prevalence_distribution.tsv",
    sep="\t",
    index=False,
)

presence_summary.to_csv(
    AUDIT_DIR / "stage04_gene_condition_presence.tsv",
    sep="\t",
    index=False,
)

filter_preview.to_csv(
    AUDIT_DIR / "stage04_gene_filter_preview.tsv",
    sep="\t",
    index=False,
)

# --------------------------------------------------
# 6. Verify no filtering has happened
# --------------------------------------------------

assert adata_pb.shape == (16, 35635)
assert adata_pb.layers["counts"].nnz == 151159
assert int(adata_pb.layers["counts"].sum()) == 12580095

assert (
    adata_pb.uns["pseudobulk_provenance"][
        "gene_filtering_applied"
    ] is False
)

print("\nPseudobulk samples removed: 0")
print("Genes actually filtered: 0")

print("\nGENE FILTER DIAGNOSTICS: PASS")


Number of genes by detected pseudobulk samples:


,n_samples_detected,n_genes
0,0,20821
1,1,1856
2,2,940
3,3,639
4,4,536
5,5,477
6,6,413
7,7,435
8,8,371
9,9,399



Gene detection across conditions:


,presence_category,n_genes
0,zero_both,20821
1,both_conditions,12002
2,ctrl_only,1690
3,stim_only,1122



Median pseudobulk library size: 721245.0
Exploratory CPM cutoff: 13.865

Exploratory gene-filtering preview:


,min_samples,cpm_cutoff,min_total_umi,n_candidate_genes,n_not_selected
0,2,13.865,15,5175,30460
1,4,13.865,15,4538,31097
2,8,13.865,15,3697,31938



Pseudobulk samples removed: 0
Genes actually filtered: 0

GENE FILTER DIAGNOSTICS: PASS


## 4. Condition-group expression filtering

Apply the condition-group CPM rule before testing differential expression.


In [8]:
import numpy as np
import pandas as pd
import json

# --------------------------------------------------
# 1. Verify unfiltered pseudobulk input
# --------------------------------------------------

assert adata_pb.shape == (16, 35635)
assert int(adata_pb.layers["counts"].sum()) == 12580095

assert count_array.shape == (16, 35635)
assert len(lib_sizes) == 16

np.testing.assert_array_equal(
    count_array.sum(axis=1),
    lib_sizes,
)

# --------------------------------------------------
# 2. Record filtering parameters
# --------------------------------------------------

min_count = 10
min_total_count = 15

group_sizes = (
    adata_pb.obs["condition"]
    .astype(str)
    .value_counts()
)

assert group_sizes["ctrl"] == 8
assert group_sizes["stim"] == 8

min_samples = int(group_sizes.min())
assert min_samples == 8

median_lib_size = float(np.median(lib_sizes))

cpm_threshold = (
    min_count / median_lib_size * 1e6
)

# --------------------------------------------------
# 3. Compute filtering mask
# --------------------------------------------------

cpm_for_filter = (
    count_array.astype(np.float64)
    / lib_sizes[:, None]
    * 1e6
)

n_samples_pass = (
    cpm_for_filter >= cpm_threshold
).sum(axis=0)

gene_total_umi = count_array.sum(axis=0)

keep_gene = (
    (n_samples_pass >= min_samples)
    & (gene_total_umi >= min_total_count)
)

assert keep_gene.shape == (35635,)
assert keep_gene.dtype == bool

# Must match previously reported preview
assert int(keep_gene.sum()) == 3697
assert int((~keep_gene).sum()) == 31938

# Verify exact match with earlier diagnostic
np.testing.assert_array_equal(
    n_samples_pass,
    gene_diag[
        "n_samples_above_cpm_cutoff"
    ].to_numpy(),
)

# --------------------------------------------------
# 4. Build per-gene audit
# --------------------------------------------------

gene_filter_audit = gene_diag.copy()

assert np.array_equal(
    gene_filter_audit["gene_id"].to_numpy(),
    adata_pb.var_names.astype(str).to_numpy(),
)

gene_filter_audit["keep_for_de"] = keep_gene

gene_filter_audit[
    "n_samples_pass_cpm"
] = n_samples_pass

gene_filter_audit[
    "pass_min_total_count"
] = gene_total_umi >= min_total_count

# --------------------------------------------------
# 5. Save selection and policy
# --------------------------------------------------

gene_filter_audit.to_csv(
    AUDIT_DIR / "stage04_gene_filter_audit.tsv.gz",
    sep="\t",
    index=False,
)

filter_policy = {
    "status": "pre_DE_filter_selected",
    "reference_method": "edgeR filterByExpr",
    "implementation": "Python equivalent core thresholds",
    "grouping_for_filter": "condition",
    "min_samples": 8,
    "min_count": 10,
    "min_total_count": 15,
    "median_unfiltered_library_size": median_lib_size,
    "cpm_threshold": cpm_threshold,
    "n_genes_before": 35635,
    "n_genes_selected": 3697,
    "uses_fold_change_or_p_values": False,
    "requires_detection_in_both_conditions": False,
    "verification_pending": (
        "Compare gene mask with actual edgeR filterByExpr "
        "before DE inference."
    ),
}

(
    AUDIT_DIR / "stage04_gene_filter_policy.json"
).write_text(
    json.dumps(filter_policy, indent=2) + "\n",
    encoding="utf-8",
)

# --------------------------------------------------
# 6. Report
# --------------------------------------------------

print("Minimum condition-group size:", min_samples)
print("Median original library size:", median_lib_size)
print("CPM cutoff:", round(cpm_threshold, 4))

print("\nGenes before filtering:", len(keep_gene))
print("Genes selected:", int(keep_gene.sum()))
print("Genes not selected:", int((~keep_gene).sum()))

print("\nGENE FILTER SELECTION: PASS")

Minimum condition-group size: 8
Median original library size: 721245.0
CPM cutoff: 13.8649

Genes before filtering: 35635
Genes selected: 3697
Genes not selected: 31938

GENE FILTER SELECTION: PASS


In [9]:
from scipy import sparse

# --------------------------------------------------
# 1. Create independent gene-filtered object
# --------------------------------------------------

adata_pb_filtered = adata_pb[:, keep_gene].copy()

assert not adata_pb_filtered.is_view

assert adata_pb_filtered.shape == (16, 3697)

# --------------------------------------------------
# 2. Verify exact selected counts
# --------------------------------------------------

filtered_counts = adata_pb_filtered.layers["counts"]

expected_counts = adata_pb.layers["counts"][
    :, keep_gene
]

assert sparse.isspmatrix_csr(filtered_counts)
assert filtered_counts.dtype == np.int64

assert (
    filtered_counts != expected_counts
).nnz == 0

assert (
    adata_pb_filtered.X != filtered_counts
).nnz == 0

assert adata_pb_filtered.raw is None

# --------------------------------------------------
# 3. Verify sample and gene identities
# --------------------------------------------------

assert adata_pb_filtered.obs_names.equals(
    adata_pb.obs_names
)

assert adata_pb_filtered.var_names.equals(
    adata_pb.var_names[keep_gene]
)

pd.testing.assert_frame_equal(
    adata_pb_filtered.var,
    adata_pb.var.iloc[np.flatnonzero(keep_gene)],
    check_exact=True,
)

# --------------------------------------------------
# 4. Record both library sizes explicitly
# --------------------------------------------------

original_library_sizes = (
    adata_pb.obs["library_size_umi"]
    .to_numpy(dtype=np.int64)
)

filtered_library_sizes = np.asarray(
    filtered_counts.sum(axis=1)
).ravel().astype(np.int64)

# Rename original field to prevent ambiguity
adata_pb_filtered.obs.rename(
    columns={
        "library_size_umi":
        "library_size_umi_unfiltered"
    },
    inplace=True,
)

adata_pb_filtered.obs[
    "library_size_umi_filtered"
] = filtered_library_sizes

np.testing.assert_array_equal(
    adata_pb_filtered.obs[
        "library_size_umi_unfiltered"
    ].to_numpy(dtype=np.int64),
    original_library_sizes,
)

assert (filtered_library_sizes > 0).all()

assert (
    filtered_library_sizes <= original_library_sizes
).all()

# --------------------------------------------------
# 5. Record filter status
# --------------------------------------------------

adata_pb_filtered.uns[
    "pseudobulk_provenance"
]["gene_filtering_applied"] = True

adata_pb_filtered.uns["gene_filter"] = (
    filter_policy.copy()
)

adata_pb_filtered.uns[
    "pseudobulk_provenance"
]["status"] = "gene_filtered_pre_DE"

# --------------------------------------------------
# 6. Verify all original objects remain unchanged
# --------------------------------------------------

assert adata_pb.shape == (16, 35635)
assert adata_pb.layers["counts"].nnz == 151159
assert int(adata_pb.layers["counts"].sum()) == 12580095

assert (
    adata_pb.uns["pseudobulk_provenance"][
        "gene_filtering_applied"
    ] is False
)

assert adata_cd14.shape == (5357, 35635)
assert int(
    adata_cd14.layers["counts"].sum()
) == 12580095

# --------------------------------------------------
# 7. Report filtered matrix properties
# --------------------------------------------------

filtered_total_umi = int(filtered_counts.sum())

print("Filtered pseudobulk shape:",
      adata_pb_filtered.shape)

print("Filtered nonzero entries:",
      filtered_counts.nnz)

print("Original total UMI:",
      int(adata_pb.layers["counts"].sum()))

print("Filtered total UMI:",
      filtered_total_umi)

print(
    "UMI retained (%):",
    round(
        100 * filtered_total_umi / 12580095,
        2
    ),
)

print("\nLibrary size before and after gene filtering:")

display(
    adata_pb_filtered.obs[
        [
            "donor_id",
            "condition",
            "library_size_umi_unfiltered",
            "library_size_umi_filtered",
        ]
    ]
)

print("\nPseudobulk samples removed: 0")
print("Original pseudobulk object unchanged: YES")
print("\nGENE-FILTERED PSEUDOBULK INTEGRITY: PASS")

Filtered pseudobulk shape: (16, 3697)
Filtered nonzero entries: 58865
Original total UMI: 12580095
Filtered total UMI: 12177533
UMI retained (%): 96.8

Library size before and after gene filtering:


,donor_id,condition,library_size_umi_unfiltered,library_size_umi_filtered
sample_id,,,,
101__ctrl,101,ctrl,524723,502847
101__stim,101,stim,726512,706853
107__ctrl,107,ctrl,469563,452578
107__stim,107,stim,436403,425918
1015__ctrl,1015,ctrl,1685517,1621180
1015__stim,1015,stim,1681881,1639091
1016__ctrl,1016,ctrl,721958,692297
1016__stim,1016,stim,707388,687282
1039__ctrl,1039,ctrl,270440,260215



Pseudobulk samples removed: 0
Original pseudobulk object unchanged: YES

GENE-FILTERED PSEUDOBULK INTEGRITY: PASS


## 5. Save and validate pseudobulk artifacts

Verify that both matrices round-trip and that the filtered matrix is an exact gene subset.


In [10]:

from pathlib import Path
import anndata as ad
import numpy as np
from scipy import sparse

# --------------------------------------------------
# 1. Define independent output paths
# --------------------------------------------------

PB_FULL_PATH = (
    PROCESSED_DIR
    / "ifnb_cd14_pseudobulk_unfiltered.h5ad"
)

PB_FILTERED_PATH = (
    PROCESSED_DIR
    / "ifnb_cd14_pseudobulk_gene_filtered.h5ad"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert PB_FULL_PATH != PB_FILTERED_PATH

# Protect the Stage 03 input file
assert PB_FULL_PATH.resolve() != H5AD_PATH.resolve()
assert PB_FILTERED_PATH.resolve() != H5AD_PATH.resolve()

# --------------------------------------------------
# 2. Verify both objects before writing
# --------------------------------------------------

assert adata_pb.shape == (16, 35635)
assert adata_pb_filtered.shape == (16, 3697)

for obj in [adata_pb, adata_pb_filtered]:

    assert sparse.isspmatrix_csr(obj.X)

    assert sparse.isspmatrix_csr(
        obj.layers["counts"]
    )

    assert obj.X.dtype == np.int64

    assert obj.layers["counts"].dtype == np.int64

    assert (
        obj.X != obj.layers["counts"]
    ).nnz == 0

    assert obj.obs_names.is_unique
    assert obj.var_names.is_unique

assert int(adata_pb.X.sum()) == 12580095
assert int(adata_pb_filtered.X.sum()) == 12177533

# --------------------------------------------------
# 3. Write two H5AD artifacts
# --------------------------------------------------

artifacts = {
    "unfiltered": (adata_pb, PB_FULL_PATH),
    "gene_filtered": (
        adata_pb_filtered,
        PB_FILTERED_PATH,
    ),
}

for label, (obj, path) in artifacts.items():

    obj.write_h5ad(
        path,
        compression="gzip",
        convert_strings_to_categoricals=False,
    )

    assert path.is_file()
    assert path.stat().st_size > 0

    print(
        label,
        "saved:",
        path,
    )

    print(
        "Size (MB):",
        round(path.stat().st_size / 1e6, 2),
    )

# --------------------------------------------------
# 4. Confirm original objects remain intact
# --------------------------------------------------

assert adata_cd14.shape == (5357, 35635)
assert int(adata_cd14.X.sum()) == 12580095

assert adata_pb.shape == (16, 35635)
assert int(adata_pb.X.sum()) == 12580095

assert adata_pb_filtered.shape == (16, 3697)

print("\nPSEUDOBULK H5AD WRITE: PASS")


unfiltered saved: /kaggle/working/kang_ifnb/data/processed/ifnb_cd14_pseudobulk_unfiltered.h5ad
Size (MB): 3.2
gene_filtered saved: /kaggle/working/kang_ifnb/data/processed/ifnb_cd14_pseudobulk_gene_filtered.h5ad
Size (MB): 0.68

PSEUDOBULK H5AD WRITE: PASS


In [11]:

import hashlib
import json
import pandas as pd
import anndata as ad
from scipy import sparse
from importlib.metadata import version

# --------------------------------------------------
# 1. Reload both saved artifacts
# --------------------------------------------------

pb_full_check = ad.read_h5ad(PB_FULL_PATH)

pb_filtered_check = ad.read_h5ad(
    PB_FILTERED_PATH
)

# --------------------------------------------------
# 2. Define exact round-trip checks
# --------------------------------------------------

def verify_roundtrip(original, loaded, label):

    assert loaded.shape == original.shape

    assert loaded.obs_names.equals(
        original.obs_names
    )

    assert loaded.var_names.equals(
        original.var_names
    )

    assert loaded.raw is None

    for matrix_name in ["X", "counts"]:

        if matrix_name == "X":
            before = original.X
            after = loaded.X
        else:
            before = original.layers["counts"]
            after = loaded.layers["counts"]

        assert sparse.isspmatrix_csr(after)
        assert after.dtype == np.int64
        assert after.nnz == before.nnz
        assert (after != before).nnz == 0

    # Compare original metadata values exactly.
    # Avoid false failures caused by categorical
    # representation differences.
    pd.testing.assert_frame_equal(
        loaded.obs.astype(object),
        original.obs.astype(object),
        check_dtype=False,
        check_exact=True,
    )

    pd.testing.assert_frame_equal(
        loaded.var.astype(object),
        original.var.astype(object),
        check_dtype=False,
        check_exact=True,
    )

    assert (
        loaded.obs.columns.equals(
            original.obs.columns
        )
    )

    assert (
        loaded.var.columns.equals(
            original.var.columns
        )
    )

    print(label, ": EXACT ROUND-TRIP MATCH")


verify_roundtrip(
    adata_pb,
    pb_full_check,
    "Unfiltered pseudobulk",
)

verify_roundtrip(
    adata_pb_filtered,
    pb_filtered_check,
    "Gene-filtered pseudobulk",
)

# --------------------------------------------------
# 3. Verify filtered genes are the exact subset
# --------------------------------------------------

selected_positions = (
    pb_full_check.var_names.get_indexer(
        pb_filtered_check.var_names
    )
)

assert len(selected_positions) == 3697
assert (selected_positions >= 0).all()
assert np.array_equal(
    selected_positions,
    np.flatnonzero(keep_gene),
)

expected_filtered = (
    pb_full_check.layers["counts"][
        :, selected_positions
    ]
)

actual_filtered = (
    pb_filtered_check.layers["counts"]
)

assert (
    actual_filtered != expected_filtered
).nnz == 0

assert pb_filtered_check.obs_names.equals(
    pb_full_check.obs_names
)

print("Selected 3,697 genes: EXACT SUBSET MATCH")

# --------------------------------------------------
# 4. Verify library-size metadata
# --------------------------------------------------

full_lib = np.asarray(
    pb_full_check.layers["counts"].sum(axis=1)
).ravel().astype(np.int64)

filtered_lib = np.asarray(
    pb_filtered_check.layers["counts"].sum(axis=1)
).ravel().astype(np.int64)

np.testing.assert_array_equal(
    full_lib,
    pb_full_check.obs[
        "library_size_umi"
    ].to_numpy(dtype=np.int64),
)

np.testing.assert_array_equal(
    full_lib,
    pb_filtered_check.obs[
        "library_size_umi_unfiltered"
    ].to_numpy(dtype=np.int64),
)

np.testing.assert_array_equal(
    filtered_lib,
    pb_filtered_check.obs[
        "library_size_umi_filtered"
    ].to_numpy(dtype=np.int64),
)

assert (filtered_lib <= full_lib).all()

print("Original and filtered library sizes: PASS")

# --------------------------------------------------
# 5. Verify paired donor samples
# --------------------------------------------------

for obj in [pb_full_check, pb_filtered_check]:

    pairs = obj.obs.pivot(
        index="donor_id",
        columns="condition",
        values="n_cells",
    )

    assert pairs.shape == (8, 2)
    assert (pairs > 0).all().all()

    assert int(obj.obs["n_cells"].sum()) == 5357

assert int(pb_full_check.X.sum()) == 12580095
assert int(pb_filtered_check.X.sum()) == 12177533

# --------------------------------------------------
# 6. Calculate SHA-256 file checksums
# --------------------------------------------------

def file_sha256(path):

    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


artifact_manifest = {}

for label, (obj, path) in artifacts.items():

    artifact_manifest[label] = {
        "filename": path.name,
        "sha256": file_sha256(path),
        "file_size_bytes": path.stat().st_size,
        "shape": [obj.n_obs, obj.n_vars],
        "total_umi": int(obj.layers["counts"].sum()),
        "nnz": int(obj.layers["counts"].nnz),
        "roundtrip": "PASS",
    }

# --------------------------------------------------
# 7. Save combined Stage 04 checkpoint
# --------------------------------------------------

stage04_manifest = {
    "stage": "04_pseudobulk",
    "anndata_version": version("anndata"),
    "source_cohort": H5AD_PATH.name,
    "source_cohort_sha256": actual_sha256,

    "n_paired_donors": 8,
    "n_pseudobulk_samples": 16,

    "aggregation": "sum_raw_integer_umi",

    "gene_filter": {
        "grouping": "condition",
        "min_samples": 8,
        "min_count": 10,
        "min_total_count": 15,
        "cpm_cutoff": float(cpm_threshold),
        "selected_genes": 3697,
        "edger_r_comparison": "pending",
    },

    "artifacts": artifact_manifest,
    "checkpoint": "PASS",
}

manifest_path = (
    AUDIT_DIR
    / "stage04_pseudobulk_roundtrip.json"
)

manifest_path.write_text(
    json.dumps(
        stage04_manifest,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

# --------------------------------------------------
# 8. Final report
# --------------------------------------------------

print("\nStage 04 artifact checksums:")

for label, info in artifact_manifest.items():
    print(label, "SHA256:", info["sha256"])

print("\nFull pseudobulk:", pb_full_check.shape)
print("Filtered pseudobulk:",
      pb_filtered_check.shape)

print("Full UMI:", int(pb_full_check.X.sum()))
print("Filtered UMI:",
      int(pb_filtered_check.X.sum()))

print("Manifest:", manifest_path)

print("\nPSEUDOBULK H5AD ROUNDTRIP: PASS")


Unfiltered pseudobulk : EXACT ROUND-TRIP MATCH
Gene-filtered pseudobulk : EXACT ROUND-TRIP MATCH
Selected 3,697 genes: EXACT SUBSET MATCH
Original and filtered library sizes: PASS

Stage 04 artifact checksums:
unfiltered SHA256: 3e252d323075a2c0ac81b0150b4169f3ecd37f43ed9461880457fc709d997e2d
gene_filtered SHA256: 3e72d3088185890d80d5ab519ce55fd9fa7731860c0f5bc125ecb773e5805cd2

Full pseudobulk: (16, 35635)
Filtered pseudobulk: (16, 3697)
Full UMI: 12580095
Filtered UMI: 12177533
Manifest: /kaggle/working/kang_ifnb/results/audit/stage04_pseudobulk_roundtrip.json

PSEUDOBULK H5AD ROUNDTRIP: PASS
